# **Week 5: Feature Selection and Regularization on the Ames Housing Dataset**

In last week’s homework, you built and evaluated linear regression models with increasingly rich feature sets. This week, we take the next critical step: **deciding which features to keep—and how much complexity is too much**.

In real-world modeling, adding more predictors does not automatically lead to better models. Extra features can increase variance, reduce interpretability, and hurt generalization. This homework explores two complementary strategies for controlling model complexity:

* **Feature selection**, where we explicitly choose a subset of predictors
* **Regularization**, where we keep all predictors but constrain their influence

Specifically, you will experiment with:

* **Forward Selection**
* **Backward Selection**
* **Ridge Regression**

Forward and backward selection approach the same problem from opposite directions—one builds a model up feature by feature, while the other prunes features away—and because both are greedy algorithms, they may arrive at different solutions. In contrast, ridge regression does not discard features at all; instead, it keeps all predictors but shrinks their coefficients toward zero, reducing model complexity in a continuous rather than discrete way.

Comparing these methods highlights an important modeling choice: whether to simplify a model by selecting features explicitly or by regularizing their influence, and how these different strategies affect generalization performance.

There are **5 problems** with **13 graded answers** worth **4 points each**, and you receive 3 points for free if you complete the whole homework.



In [1]:
# Useful imports

import os
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import io
import zipfile

from sklearn.model_selection   import train_test_split, cross_val_score,RepeatedKFold
from sklearn.linear_model      import LinearRegression,Ridge,Lasso
from sklearn.model_selection   import GridSearchCV
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics           import mean_squared_error, r2_score
from sklearn.preprocessing     import StandardScaler 

random_seed = 42

## **Prelude: Download the Preprocessed Ames Housing Dataset**

I have stored the dataset as a zipped directory containing the following four files from an 80%-20% split of the Ames dataset after preprocessing (as done in the last homework):

- `X_train.csv`         
- `y_train.csv`
- `X_test.csv`          
- `y_test.csv`

**TODO:** Run the following cell to download and extract the dataset into the current working directory of this notebook.  
Alternatively, you can manually download the zip file from the provided URL, extract it, and place the files in the same directory as your notebook.

> ⚠️ **Important:**  
> DO NOT use your own version of these datasets from the last homework. The autograder expects the exact split provided in my files for both training and testing sets. Using any other split will result in incorrect results during grading.


In [2]:
# Download the Ames Housing Dataset from Snyder's web site

# URL to the zip file
zip_url = "https://www.cs.bu.edu/fac/snyder/cs505/Data/ames_housing.zip"

try:
    response = requests.get(zip_url)
    response.raise_for_status()  # Raise an error for bad status codes
    with zipfile.ZipFile(io.BytesIO(response.content)) as zipf:
        zipf.extractall("Ames_Dataset_HW")
    print("Files downloaded and extracted successfully.")
except requests.exceptions.RequestException as e:
    print(f"Error downloading the file: {e}")


DATA_DIR = "Ames_Dataset_HW"

X_train = pd.read_csv(f"{DATA_DIR}/X_train.csv")
X_test  = pd.read_csv(f"{DATA_DIR}/X_test.csv")

y_train = pd.read_csv(f"{DATA_DIR}/y_train.csv").squeeze("columns")
y_test  = pd.read_csv(f"{DATA_DIR}/y_test.csv").squeeze("columns")

print("Training and testing datasets loaded successfully.")



Files downloaded and extracted successfully.
Training and testing datasets loaded successfully.


## **Problem One: Forward Selection with the Ames Dataset**

For our first experiment we will apply the forward feature selection algorithm to the dataset. Follow these steps, using the notebook from this week's video lesson as a resource as necessary. 

**If you have not looked at the video yet, please do it now before continuing.**

**Steps to Follow:**
1. **Observe** that we have provided the `forward_feature_selection` algorithm from the video notebook for you to use. 

2. **Copy** from the video notebook into the cell indicated below the code that runs forward feature selection and generates a plot (excluding the part that prints the test MSE--we'll do that at the end of this homework). Or write your own version!

3. **Modify** this code to display the **Root Mean Squared Error (RMSE)** instead of MSE.  
   - *Remember*: RMSE is the square root of MSE, which provides results in dollars rather than dollars squared, making the metric more interpretable.  
   - Both the plot of RMSE vs. the number of selected features and the printout of the Best CV Score should use RMSE.

4. **Run** the code on `X_train` and `y_train` (keeping all default settings) to:
   - Generate the plot of **Root Mean Squared Error (RMSE)** vs. **Features Added**, and  
   - Print the **Best Feature Set** found and the **Best CV RMSE Score**.

**Hints:**
- The feature names will be more legible if you increase the size of the plot and change the angle and size of the xticks, e.g.,

        plt.xticks(range(1, len(selected_features_forward) + 1), selected_features_forward, rotation=60, ha='right', fontsize=6) 

- You may wish to change the scale of the y-axis to better see the behavior around the minimum point.

In [3]:
# Forward Feature Selection

def forward_feature_selection(X, y, model, 
                              scoring='neg_mean_squared_error', 
                              cv = 5, 
                              tol=None,               # None = no delta cutoff
                                                      # use 0.0 for "no further improvements"
                                                      # and 1e-4 for "point of diminishing returns"                                      
                              max_features=None,      # None = use all features
                              n_jobs=-1,
                              verbose=False
                             ):
    selected_features = []                            # List to store the order of features selected
    remaining_features = list(X.columns)              # Features not yet selected
    best_scores = []                                  # List to store the CV score after each feature addition
    previous_score = float('inf')                     # Initialize previous score for improvement comparison

    # Track the best subset of features and its corresponding score
    
    best_feature_set = None                           # Best combination of features found so far
    best_score = float('inf')                         # Best CV score observed so far
    
    while remaining_features:
        scores = {}                                   # Dictionary to hold CV scores for each candidate feature
        for feature in remaining_features:
            current_features = selected_features + [feature]
            
            # Compute the CV score for the current set of features (negated MSE, so lower is better)
            cv_score = -cross_val_score(model, X[current_features], y, 
                                        scoring=scoring, cv=cv, n_jobs=n_jobs
                                       ).mean()
            scores[feature] = cv_score

        # Select the feature that minimizes the CV score
        best_feature = min(scores, key=scores.get)
        current_score = scores[best_feature]
            
        # Check if the improvement is significant based on the tolerance (tol)
        if tol is not None and previous_score - current_score < tol:
            if verbose:
                print("Stopping early due to minimal improvement.")
            break

        # Add the best feature to the selected list and update score trackers
        selected_features.append(best_feature)
        best_scores.append(current_score)
        remaining_features.remove(best_feature)
        previous_score = current_score

        if verbose:
            print(f"\nFeatures: {selected_features[-3:]}, CV Score (MSE): {current_score:.4f}")
        
        # Update the best subset if the current score is better than the best so far
        if current_score < best_score:
            best_score = current_score
            best_feature_set = selected_features.copy()
        
        # Check if the maximum number of features has been reached
        if max_features is not None and len(selected_features) >= max_features:
            break

    return (
        selected_features,      # List of features in the order they were selected (this will be ALL features if max_features == None
        best_scores,            # List of cross-validation scores corresponding to each addition in the previous list
        best_feature_set,       # The subset of features that achieved the best CV score.
        best_score              # The best CV score
    )


In [4]:
# Your code here:  Run Forward Feature Selection, plot the results, and print out the Best Feature Set and the Best CV Score found. 



### Problem One Graded Questions

Assign `a1a` to the number of features in the best feature set found

In [5]:
# Your answer here; use an expression, not a constant derived by examining the data

a1a = 0                     # replace 0 with an expression

In [6]:
# Do not change this cell in any way

print(f"a1a = {a1a}")

a1a = 0


Assign `a1b` to the best CV RMSE score found.

In [7]:
# Your answer here; use an expression, not a constant derived by examining the data

a1b = 0                     # replace 0 with an expression

In [8]:
# Do not change this cell in any way

print(f"a1b = ${a1b:,.2f}")

a1b = $0.00


## **Problem Two: Backward Selection with the Ames Dataset**

Now, repeat the same process as in Problem One, but using the `backward_feature_selection` algorithm from this week’s video notebook. Again, we will use 5-Fold CV scoring. 

**Steps to Follow:**
1. **Observe** that we have provided the `backward_feature_selection` algorithm from this week's video notebook for you to use.
2. **Run** the backward selection algorithm on the Ames dataset (`X_train` and `y_train`).
3. **Plot** the results: Display the Root Mean Squared Error (RMSE) vs. the features removed by the algorithm.
4. **Print** the Best Feature Set found by backward selection and the corresponding best CV RMSE Score.


In [9]:
# Backward Feature Selection

def backward_feature_selection(X, y, model, 
                               scoring='neg_mean_squared_error', 
                               cv = 5, 
                               tol=None,               # None = no delta cutoff
                                                       # use 0.0 for "no further improvements"
                                                       # and 1e-4 for "point of diminishing returns"                                      
                               max_features=None,      # If None, remove features until only 1 remains
                                                       # Otherwise, stop when this many features remain
                               n_jobs=-1,
                               verbose=True
                              ):
    
    # Helper function to compute CV score using LinearRegression
    def cv_score(features):
        return -cross_val_score(model, X[features], y, 
                                scoring=scoring, cv=cv, 
                                n_jobs=n_jobs          ).mean()
    
    # Start with all features (using a list for easier manipulation)
    features_remaining = list(X.columns)
    
    # Compute initial CV score with the full feature set
    initial_score = cv_score(features_remaining)
    
    # Initialize tracking variables
    best_score        = initial_score                # Best (lowest) CV score observed so far
    best_feature_set  = features_remaining.copy()    # Feature set corresponding to best_score
    selected_features = ['NONE']                     # List to record the order in which features are removed
    best_scores       = [initial_score]              # List to record the CV score after each removal (starting with full set)
    
    if verbose:
        print("Start with full set of features:")
        print(f'{features_remaining}  CV score (MSE): {np.around(initial_score, 4)}\n')
    
    # Determine the target number of features to keep:
    # For backward elimination, if max_features is None, we remove until 1 feature remains.
    target_feature_count = 1 if max_features is None else max_features
    
    prev_score = initial_score
    round_num = 1
    # Continue removing features until we reach the target count
    while len(features_remaining) > target_feature_count:
        if verbose:
            print(f'Round {round_num}:')
            
        # Initialize variables to track the best removal in this round
        lowest_score = float('inf')
        feature_to_remove = None
        best_new_features = None
        
        # Try removing each feature one at a time
        for feature in features_remaining:
            new_feature_set = features_remaining.copy()
            new_feature_set.remove(feature)
            new_score = cv_score(new_feature_set)
            if verbose:
                print('Trying removal of:',feature, np.around(new_score, 4))
            if new_score < lowest_score:
                lowest_score = new_score
                feature_to_remove = feature
                best_new_features = new_feature_set
        
        # Check if improvement is significant enough (if tol is set)
        if tol is not None and (prev_score - lowest_score) < tol:
            if verbose:
                print("\nStopping early due to minimal improvement.")
            break
        
        # Update the best score and feature set if current removal improves performance
        if lowest_score < best_score:
            best_score = lowest_score
            best_feature_set = best_new_features.copy()
        
        # Update trackers for this round
        prev_score = lowest_score
        features_remaining = best_new_features
        selected_features.append(feature_to_remove)
        best_scores.append(lowest_score)
        
        if verbose:
            print(f'\nRemoving {feature_to_remove}:  CV score (MSE) {np.around(lowest_score, 4)}\n')
        round_num += 1

    return (
        selected_features,      # Order in which features were removed
        best_scores,            # CV scores after each removal step
        best_feature_set,       # Feature set that achieved the best CV score
        best_score              # Best (lowest) CV score
    )


In [10]:
# Your code here:  Run Backward Feature Selection, plot the results, and print out the Best Feature Set and the Best CV RMSE Score found. 


### Problem Two Graded Questions

Assign `a2a` to the number of features in the best feature set found

In [11]:
# Your answer here; use an expression, not a constant derived by examining the data

a2a = 0                     # replace 0 with an expression

In [12]:
# Do not change this cell in any way

print(f"a2a = {a2a}")

a2a = 0


Assign `a2b` to the best CV RMSE score found.

In [13]:
# Your answer here; use an expression, not a constant derived by examining the data

a2b = 0                     # replace 0 with an expression

In [14]:
# Do not change this cell in any way

print(f"a2b = ${a2b:,.2f}")

a2b = $0.00


## **Problem Three: Ridge Regression on the Ames Housing Dataset**

In this problem, we will apply Ridge Regression to the Ames Housing dataset. Ridge Regression includes a hyperparameter $\alpha$ that controls the strength of the regularization penalty, which helps prevent overfitting by constraining the growth of the model’s coefficients. A higher $\alpha$ penalizes large coefficients more, while a lower $\alpha$ allows them to grow larger.

When creating the model, the parameter must be specified, for example:

```python
ridge_model = Ridge(alpha=100)
```

This introduces another instance of the model selection problem: we must determine the value of $\alpha$ that yields the best CV RMSE score.

**Steps to Follow:**

1. **Standardize the Data:**  
   Ridge regression is sensitive to the scale of the features, so we will use `StandardScaler` to standardize the feature set to have a mean of 0 and a standard deviation of 1 before training and testing. Note that the target variable should **not** be scaled.

2. **Perform Cross-Validation Over a Range of Alpha Values:**  
   For each $\alpha \in \{100, 110, 120, \dots, 500\}$, calculate the cross-validation RMSE score for the model using 5-Fold CV scoring.  

3. **Visualize the Results:**  
   Plot the CV RMSE scores against the $\alpha$ values.

4. **Identify the Best Alpha:**  
   Determine the $\alpha$ that results in the minimum CV RMSE Score and print it out, along with the score.

**Tip:** It would be an *excellent* idea to add the suffix `_scaled` to any set to which you apply scaling.

In [15]:
# Your code here


### Problem Three Graded Questions

Set `a3a` to the alpha determined to have the minimum CV RMSE score.

In [16]:
# Your answer here; use an expression, not a constant derived by examining the data

a3a = 0                     # replace 0 with an expression

In [17]:
# Do not change this cell in any way. 

print(f"a3a = {a3a}")

a3a = 0


Set `a3b` to the CV score found at that alpha. 

In [18]:
# Your answer here; use an expression, not a constant derived by examining the data

a3b = 0                     # replace 0 with an expression

In [19]:
# Do not change this cell in any way. 

print(f"a3b = ${a3b:,.2f}")

a3b = $0.00


## **Problem Four: Final Evaluation on the Test Set**

In this final problem, you will evaluate the strongest feature-selection model and the strongest Ridge Regression model on the held-out test set.

### **Steps to Follow**

### 1. Identify Your Two Finalists

From your previous work:

* Select the better of your **Forward Selection** and **Backward Selection** models based on **cross-validation RMSE**.
* Select the **Ridge Regression** model (with tuned $\alpha$) with the best cross-validation RMSE.

These two models will be evaluated on the test set.

### 2. Evaluate the Feature-Selection Model

Train a Linear Regression model using `X_train` restricted to the best feature set identified by either Forward Selection or Backward Selection.

Evaluate the model on `X_test` restricted to the same feature set and compute the test RMSE.

### 3. Evaluate the Ridge Regression Model

1. Create a `StandardScaler`.
2. **Fit the scaler using only `X_train`.**
3. Use the fitted scaler to transform both `X_train` and `X_test`.
4. Train a Ridge Regression model using the best alpha found during cross-validation.
5. Evaluate the model on the transformed test set and compute the test RMSE.

> **Important:** Do not fit the scaler using the test data. The test set must remain completely unseen until the final evaluation. Fitting a scaler on the entire dataset (or on the test set) introduces **data leakage** and can produce overly optimistic performance estimates.

> **Recall:** Any preprocessing step that learns information from the data (scaling, imputation, feature selection, PCA, etc.) must be fit on the training data only and then applied to the test data.

### 4. Report Your Results

Print:

* The name of the best feature-selection model and its test RMSE.
* The best Ridge Regression alpha and its test RMSE.
* Which model had the lower **cross-validation RMSE** and was therefore selected as your final model.

### 5. Answer the graded questions. 
 

In [20]:
# Your code here


### Problem Four Graded Questions

Set `a4a` to the best model, according to the CV RMSE score.

- 1 = Forward Selection
- 2 = Backward Selection
- 3 = Ridge Regression

In [21]:
# Your answer here

a4a = 0                     # replace 0 with one of [1,2,3]

In [22]:
# Do not change this cell in any way. 

print(f"a4a = {a4a}")

a4a = 0


Set `a4b` to the test RMSE for the best model.

In [23]:
# Your answer here; use an expression, not a constant derived by examining the data

a4b = 0                     # replace 0 with an expression

In [24]:
# Do not change this cell in any way. 

print(f"a4b = ${a4b:,.2f}")

a4b = $0.00


## **Problem Five: Analytical Questions**

Select the single **best** answer to each of the following multiple choice questions on this week's material and a review about a tricky issue from last week ("data leakage").  

### Part A: Ridge Regression and Feature Scaling

Why is feature scaling important before applying Ridge Regression?

1. Ridge Regression only works with categorical features.
2. Ridge Regression automatically removes irrelevant features.
3. The regularization penalty depends on coefficient size, which is affected by feature scale.
4. Scaling guarantees a lower test RMSE.

In [25]:
# Your answer here. 

a5a = 0                    # Must be one of 1, 2, 3, 4

In [26]:
### Do not change this cell in any way

print(f'a5a = {a5a}')

a5a = 0


#### Explanation

Ridge Regression penalizes large coefficient values. If features are measured on very different scales, some coefficients may appear artificially large or small simply because of the units being used. Standardizing the features places them on a common scale, allowing the regularization penalty to affect all features fairly.

### Part B: Model Selection Using Cross-Validation

Suppose two models have the following cross-validation RMSE scores:

| Model             | CV RMSE |
| ----------------- | ------: |
| Forward Selection | \$24,500 |
| Ridge Regression  | \$25,300 |

Which model should be selected for final testing?

1. Forward Selection
2. Ridge Regression
3. Both should be evaluated on the test set before deciding.
4. There is not enough information.

In [27]:
# Your answer here. 

a5b = 0                    # Must be one of 1, 2, 3, 4

In [28]:
# Do not change this cell in any way

print(f'a5b = {a5b}')

a5b = 0


### Part C: Purpose of the Test Set

Why didn't we evaluate all three models on the test set and choose the one with the lowest test RMSE?

1. The test set is usually too small to evaluate multiple models.
2. Doing so would allow information from the test set to influence model selection.
3. Cross-validation and testing always produce identical results.
4. The test set should only be used with Ridge Regression.

In [29]:
# Your answer here. 

a5c = 0                    # Must be one of 1, 2, 3, 4

In [30]:
# Do not change this cell in any way

print(f'a5c = {a5c}')

a5c = 0


### Part D: Data Leakage During Imputation

Suppose a dataset contains missing values, and you decide to replace them using the median of the feature. Which procedure best avoids data leakage?

1. Compute the median using the entire dataset before creating the train/test split.
2. Compute the median separately for the training set and test set.
3. Compute the median using only the training set, then use that same value to fill missing values in both the training and test sets.
4. Compute the median using only the test set.

In [31]:
# Your answer here. 

a5d = 0                    # Must be one of 1, 2, 3, 4

In [32]:
# Do not change this cell in any way

print(f'a5d = {a5d}')

a5d = 0


### Part E: Pipelines and Cross-Validation

Why do we place the imputer inside a Scikit-Learn Pipeline when using cross-validation?

1. Pipelines make the code run faster.
2. Pipelines automatically choose the best imputation strategy.
3. Pipelines ensure that imputation statistics are computed using only the training data within each fold.
4. Pipelines eliminate the need for validation data.

In [33]:
# Your answer here. 

a5e = 0                   # Must be one of 1, 2, 3, 4

In [34]:
# Do not change this cell in any way

print(f'a5e = {a5e}')

a5e = 0


### Appendix: Prof Snyder's Post on Piazza from 6/2/26

Folks,   Jan mentioned an important point during the Live Session, but which is a bit to complicated for the first time we have seen validation sets. 

This post is to answer Jan's question and make you aware of some details which we are not using in Homework 4, but might be useful later on. 

The basic idea is to avoid "data leakage," which means that you have violated the "held-out" property which is the core of the framework.  Just like the best evaluation of your learning at the end of the course is a final exam for you which

you have no knowlege of what I will ask, the best test set is completely unknown to the model. Same for validation sets. 

In homework 4, we are not being too obsessive about this, since it is the first time we have seen imputation and the training/validation/test split. 

But the absolutely perfect way to do this is that any statistics used for imputation (such as a mean, median, mode, or most-frequent category) should be computed using only the training data. Those same values are then applied to the validation and test data.

For example, if we use median imputation, we first compute the median from the training set. Missing values in both the training set and the test/validation set are then replaced using that same median. We do not recompute the median using the test or validation data.

This gets a bit more complicated if you use cross-validation. To do this perfectly, for each fold, the imputation statistics are computed from the training portion of that fold and then applied to the corresponding validation portion. You need to create a pipeline and feed that as the cross-validation method:

    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LinearRegression
    
    pipeline = Pipeline(
    
         [
          ("imputer", SimpleImputer(strategy="median")),
          ("model", LinearRegression())
         ]
    
    )
    
    cross_val_score(pipeline, X, y, cv=5)